# 歌词分词，词性标注

In [ ]:
import json
import pandas as pd

# import jieba
# import jieba.posseg as pseg
import thulac
from collections import Counter
from openai import OpenAI

In [2]:
# 可以选择是否加载
# jieba.load_userdict('data/mayday_dict_simple.txt')

In [3]:
import sys
sys.path.append('..')

# 分词，词频与词性分析

In [4]:
word_to_fix = {
    '阮': 'r',
    '袂': 'v'
}

In [5]:
def process_lyrics_with_jieba(text):
    # 1. 词性标注与分词
    # jieba.posseg 会同时返回词和词性
    words_with_pos = pseg.cut(text)

    
    # 2. 过滤无意义字符（标点、空格、单字符停用词）
    filtered_data = []
    for word, pos in words_with_pos:
        # 排除标点符号（x表示标点）及空白字符
        if pos != 'x' and len(word.strip()) > 0:
            if word in word_to_fix:
                filtered_data.append((word, word_to_fix[word]))
            else:
                filtered_data.append((word, pos))
    
    # 3. 统计词频
    word_counts = Counter([item[0] for item in filtered_data])
    
    # 4. 汇总信息 (词, 词性, 频数)
    # 我们以词为 Key，存储词性
    word_pos_map = {word: pos for word, pos in filtered_data}
    
    # 排序：按词频从高到低
    sorted_results = []
    for word, count in word_counts.most_common():
        sorted_results.append({
            "word": word,
            "pos": word_pos_map[word],
            "freq": count # 词频
        })
    
    return sorted_results

In [38]:
thu = thulac.thulac(seg_only=False, filt=True) 

def process_lyrics_with_thulac(text, word_to_fix=None):
    if not text:
        return []
    
    # 2. 执行分词与词性标注
    # 返回格式为 [[word, pos], [word, pos], ...]
    words_with_pos = thu.cut(text)
    
    # 3. 过滤无意义字符与词性修正
    # thulac 的标点词性通常是 'w'
    filtered_data = []
    for word, pos in words_with_pos:
        word = word.strip()
        # 排除标点符号、空白字符
        if pos != 'w' and len(word) > 0:
            # 逻辑修正：word_to_fix 通常是修正词性
            if word_to_fix and word in word_to_fix:
                filtered_data.append((word, word_to_fix[word]))
            else:
                filtered_data.append((word, pos))
    
    # 4. 统计词频
    word_counts = Counter([item[0] for item in filtered_data])
    
    # 5. 汇总信息
    # 建立 word -> pos 映射
    word_pos_map = {word: pos for word, pos in filtered_data}
    
    sorted_results = []
    for word, count in word_counts.most_common():
        sorted_results.append({
            "word": word,
            "pos": word_pos_map[word],
            "freq": count
        })
    
    return sorted_results

Model loaded succeed


In [29]:
def process_lyrics_with_thulac(text):
    """
    使用THULAC替代jieba完成歌词分词、词性标注、词频统计
    功能与原jieba版本完全一致，输出格式保持不变
    :param text: 输入的歌词文本
    :return: 按词频排序的结果列表，元素为{"word": 词, "pos": 词性, "freq": 频数}
    """
    # 1. 词性标注与分词（THULAC核心调用）
    # cut返回格式：字符串 "词1/词性1 词2/词性2 ..."，需解析为列表
    cut_result = thu.cut(text, text=True)
    # 解析结果：拆分为(词, 词性)的列表
    words_with_pos = []
    for item in cut_result.split():
        if "/" in item:  # 确保是"词/词性"格式，过滤异常项
            word, pos = item.split("/", 1)  # 只拆分一次，避免词中含"/"的情况
            words_with_pos.append((word.strip(), pos.strip()))

    # 2. 过滤无意义字符（兼容原逻辑：排除标点、空白、应用词性修正）
    filtered_data = []
    for word, pos in words_with_pos:
        # 排除空白字符（THULAC的filt=True已过滤大部分标点，但做二次兜底）
        if len(word) > 0:
            # 应用自定义词性修正映射
            if word in word_to_fix:
                filtered_data.append((word, word_to_fix[word]))
            else:
                # THULAC的标点词性为"w"，对应jieba的"x"，此处统一过滤
                if pos != 'w':
                    filtered_data.append((word, pos))
    
    # 3. 统计词频（与原逻辑完全一致）
    word_counts = Counter([item[0] for item in filtered_data])
    
    # 4. 汇总信息 (词, 词性, 频数)（与原逻辑完全一致）
    word_pos_map = {word: pos for word, pos in filtered_data}
    
    # 排序：按词频从高到低（与原逻辑完全一致）
    sorted_results = []
    for word, count in word_counts.most_common():
        sorted_results.append({
            "word": word,
            "pos": word_pos_map[word],
            "freq": count  # 词频
        })
    
    return sorted_results

In [40]:
def lyric_words_process(path_prefix, word_to_fix=None):
    lyric_file_path = path_prefix + 'cleared_lyric_data.json'
    # 读取歌词文件
    with open(lyric_file_path, 'r') as f:
        lyric_data = json.load(f)
    lyric_words_dict = {}
    for i in lyric_data:
        if i:
            # lyric_words_dict[i['song_id']] = process_lyrics_with_jieba(
            #     i['lyrics_text'])
            lyric_words_dict[i['song_id']] = process_lyrics_with_thulac(
                i['lyrics_text'], word_to_fix=word_to_fix)
    rows = []
    for song_id, word_list in lyric_words_dict.items():
        for item in word_list:
            # 创建新字典，保留原始数据并加入歌曲ID列
            new_row = {
                'song_id': song_id,
                'word': item['word'],
                'pos': item['pos'],
                'freq': item['freq']
            }
            rows.append(new_row)

    # 3. 转换为 DataFrame
    df_word = pd.DataFrame(rows)
    return df_word

In [16]:
def words_data_merge(df_word, df_songs):
    # 合并
    # 1. 确保 df_word 的 song_id 是字符串
    df_word['song_id'] = df_word['song_id'].astype(str)

    # 2. 确保 df_unique 的 song_id 是字符串（并去掉可能存在的空格）
    df_songs['song_id'] = df_songs['song_id'].astype(str).str.strip()

    # 3. 执行合并
    df_merged = df_word.merge(df_songs, on='song_id', how='left')

    # 4. 删除空值
    # df_merged = df_merged.dropna()

    return df_merged

# main

In [51]:
file_path_prefix = "data/jaychou/"
file_path_prefix = "data/mayday/"

In [52]:
# 歌曲数据
df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_mid,duration,publish_time,song_name_unique,album_id,publish_date
0,107709592,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
1,447807,002M8hNI2QgtRY,突然好想你,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,后青春期的诗,0020I7sO0ayXhN,265,1224691200,突然好想你,36459,2008-10-23
2,5131923,003lhef916qYN2,步步,《步步惊情》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,0006MmDz4Hl2Ud,273,1388332800,步步,451706,2013-12-30
3,4830286,0033P66R0qEtlT,知足,《后来的我们》电影插曲,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,003PIMo40rxcAn,256,1124985600,知足,96397,2005-08-26
4,106528423,000aHM1h2bD5Kb,派对动物,NaN,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,249,1469030400,派对动物,1393445,2016-07-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,4996096,003Xy9E32vvMLe,入阵曲,《兰陵王》电视剧主题曲,五月天,74,000Sp0Bz4JXH0o,步步 自选作品辑,002adz882rV5uh,209,1377792000,入阵曲,451706,2013-12-30
133,4932058,003PaRAX3j5wJk,生命有一种绝对,NaN,五月天,74,000Sp0Bz4JXH0o,时光机,0015r2I31enfaR,239,1038672000,生命有一种绝对,96353,2003-11-07
134,4830242,000PoJAV4NPMzW,温柔 (还你自由版),NaN,五月天,74,000Sp0Bz4JXH0o,知足 最真杰作选,001ntd0y01uQ4g,426,1088611200,温柔 (还你自由版),96397,2005-08-26
135,4834459,002nqyCb1bUnk6,Enrich Your Life,NaN,五月天,74,000Sp0Bz4JXH0o,神的孩子都在跳舞,0006oAnx03zXUC,166,1096560000,Enrich Your Life,96368,2004-11-01


In [53]:
# 五月天需要使用word_to_fix
df_word = lyric_words_process(file_path_prefix, word_to_fix)

In [54]:
df_word

,song_id,word,pos,freq
0,107709592,来,v,12
1,107709592,能,v,6
2,107709592,人生,n,6
3,107709592,有,v,4
4,107709592,是,v,4
...,...,...,...,...
8387,519403016,天涯飞奔,id,1
8388,519403016,回头,v,1
8389,519403016,飞奔,v,1
8390,519403016,请,v,1


In [55]:
df_merged = words_data_merge(df_word, df_songs)
df_merged

,song_id,word,pos,freq,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_mid,duration,publish_time,song_name_unique,album_id,publish_date
0,107709592,来,v,12,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
1,107709592,能,v,6,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
2,107709592,人生,n,6,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
3,107709592,有,v,4,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
4,107709592,是,v,4,0022QuVR1LcRHN,后来的我们,巅峰榜2025年度榜单QQ音乐巅峰榜百大巅峰金曲,五月天,74,000Sp0Bz4JXH0o,自传,002fRO0N4FftzY,346,1469030400,后来的我们,1393445,2016-07-21
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8387,519403016,天涯飞奔,id,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265,1727625600,我不愿让你一个人,90142,2011-12-16
8388,519403016,回头,v,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265,1727625600,我不愿让你一个人,90142,2011-12-16
8389,519403016,飞奔,v,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265,1727625600,我不愿让你一个人,90142,2011-12-16
8390,519403016,请,v,1,000B69Qg0S8WUF,我不愿让你一个人,《今夜一起为爱鼓掌》电视剧插曲,五月天,74,000Sp0Bz4JXH0o,第二人生,001Jhk1t0SC1FZ,265,1727625600,我不愿让你一个人,90142,2011-12-16


In [56]:
df_merged.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)

# 测试

In [22]:
text = "我会唱歌"
cut_result = jieba.lcut(text)
print("分词结果：", cut_result) 

分词结果： ['我会', '唱歌']


In [23]:
# 测试1：看分词结果，是否整体切分为“有点”
text = "有点后悔、有点开心"
print("分词结果：", jieba.lcut(text))  # 若输出['有点','后悔','、','有点','开心']→完整词；若拆分则是正常逻辑
# 测试2：看具体词性标注
for word, flag in pseg.lcut(text):
    print(f"词：{word}，词性：{flag}")

分词结果： ['有点', '后悔', '、', '有点', '开心']
词：有点，词性：n
词：后悔，词性：v
词：、，词性：x
词：有点，词性：n
词：开心，词性：v


In [26]:
pip install thulac

Looking in indexes: https://mirrors.tencent.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 MB 2.1 MB/s  0:00:25m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


In [27]:
import thulac

# 初始化（seg_only=False：开启词性标注）
thu = thulac.thulac(seg_only=False)

# 测试
text = "有点后悔、我会唱歌"
result = thu.cut(text, text=True)  # text=True：返回字符串格式
print(result)

Model loaded succeed
有点_d 后悔_v 、_w 我_r 会_v 唱歌_v
